# M6S1 — Neuromorphic computing: a first spiking neuron
**IE University · HPC 2026–27 · One session · About 30–40 minutes**

An always-on sensor may receive many repeated readings. Can it communicate changes, and can a neuron accumulate evidence before sending an output event?

You will read a short C program, explain state/threshold/reset, change one parameter, and count sensor events. Python only writes, compiles and runs the C code. No training libraries, datasets or special hardware are needed.

These are **CPU teaching simulations**, not measurements of a neuromorphic chip. This notebook covers only neuromorphic computing. Other Module 6 topics need their own session notebooks.


## 0. Open and run
1. In SciTech JupyterLab, use **File → Open from URL**, paste the raw notebook URL from the README, then save your own copy.
2. Select Python 3 and run cells from top to bottom. Files go into a new student-owned folder under `~/hpc_m6s1_neuromorphic/`.
3. On SciTech the runner below uses `srun -p cpu -N1 -n1 -c1 --mem=1G --time=00:02:00` for each tiny execution. Wait for the job to finish. On a local computer without Slurm it runs locally.
4. Follow the [course cluster guide](../../cluster/README.md) if GCC is missing; use your course compiler environment. No package installation is needed. A queue wait or Slurm error is not an experimental result.

**Status:** PASS means an output check succeeded; FAIL means execution or a check failed; SKIPPED means GCC was unavailable; NOT RUN means a cell has not been executed. A local run does not validate SciTech. Run All resets results; parameter cells can be rerun separately.


In [1]:
from pathlib import Path
import os, shutil, subprocess, tempfile

# Create a fresh student-owned directory for this run.
base = Path.home() / "hpc_m6s1_neuromorphic"
base.mkdir(exist_ok=True)
work = Path(tempfile.mkdtemp(prefix="run_", dir=base))
print("Working folder:", work)
print("Execution:", "SCITECH_SLURM (not yet run)" if shutil.which("srun") else "LOCAL_CPU")

# This small helper keeps compilation commands out of each experiment.
def run_c(name, *parameters):
    if not shutil.which("gcc"):
        print("SKIPPED: GCC unavailable. Load the course compiler and rerun.")
        return None
    compile_result = subprocess.run(["gcc", "-std=c11", "-Wall", "-Wextra", "-Werror", str(work / (name + ".c")), "-o", str(work / name)], capture_output=True, text=True)
    if compile_result.returncode:
        raise RuntimeError("FAIL: compilation\n" + compile_result.stderr)
    command = [str(work / name), *map(str, parameters)]
    env = os.environ.copy()
    if shutil.which("srun"):
        for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_NODE", "SLURM_MEM_PER_GPU"):
            env.pop(key, None)  # avoid conflicting inherited JupyterHub memory settings
        command = ["srun", "-p", "cpu", "-N1", "-n1", "-c1", "--mem=1G", "--time=00:02:00"] + command
    result = subprocess.run(command, capture_output=True, text=True, env=env)
    if result.returncode:
        raise RuntimeError("FAIL: execution; inspect the scheduler/compiler message\n" + result.stderr)
    print(result.stdout)
    return result.stdout


Working folder: /root/hpc_m6s1_neuromorphic/run_zylz8868
Execution: LOCAL_CPU


## 1. Make a neuron spike
**Problem:** a sensor should trigger an alert only after enough evidence accumulates.

**Predict:** input adds 0.6 each step. We keep 80% of the previous state. With threshold 1.0, will the first input produce a spike?

Read these three rules:
- `state = retention * state + input`: forget some old evidence, then add new evidence.
- `state >= threshold`: decide whether to emit a spike.
- `state = 0.0`: reset after the spike.

This is a simplified discrete leaky integrate-and-fire model with dimensionless values, constant input, fixed weights and reset to zero. It does not learn. Each printed row is one simulation step.


In [2]:
neuron_c = r"""#include <stdio.h>
#include <stdlib.h>
#include <math.h>

int main(int argc, char **argv) {
    if (argc != 3) return 1;  // arguments: threshold retention
    double threshold = atof(argv[1]);
    double retention = atof(argv[2]);
    if (!isfinite(threshold) || threshold <= 0 ||
        !isfinite(retention) || retention < 0 || retention > 1) return 1;
    double state = 0.0;
    const double input = 0.6;
    int spikes = 0;
    printf("step  before_reset  spike  after_reset\n");
    for (int step = 1; step <= 8; step++) {
        state = retention * state + input;  // leak, then integrate
        double before = state;
        int spike = state >= threshold;    // threshold test
        if (spike) {
            spikes++;
            state = 0.0;                   // reset
        }
        printf("%4d  %12.3f  %5d  %11.3f\n", step, before, spike, state);
    }
    printf("TOTAL_SPIKES=%d\n", spikes);
    return 0;
}
"""
(work / "neuron.c").write_text(neuron_c)
print(neuron_c)


#include <stdio.h>
#include <stdlib.h>
#include <math.h>

int main(int argc, char **argv) {
    if (argc != 3) return 1;  // arguments: threshold retention
    double threshold = atof(argv[1]);
    double retention = atof(argv[2]);
    if (!isfinite(threshold) || threshold <= 0 ||
        !isfinite(retention) || retention < 0 || retention > 1) return 1;
    double state = 0.0;
    const double input = 0.6;
    int spikes = 0;
    printf("step  before_reset  spike  after_reset\n");
    for (int step = 1; step <= 8; step++) {
        state = retention * state + input;  // leak, then integrate
        double before = state;
        int spike = state >= threshold;    // threshold test
        if (spike) {
            spikes++;
            state = 0.0;                   // reset
        }
        printf("%4d  %12.3f  %5d  %11.3f\n", step, before, spike, state);
    }
    printf("TOTAL_SPIKES=%d\n", spikes);
    return 0;
}



In [3]:
# Run: threshold 1.0; retention 0.8
baseline = run_c("neuron", 1.0, 0.8)
if baseline is not None:
    assert "TOTAL_SPIKES=4" in baseline, "FAIL: expected four spikes"
    print("PASS: spikes occur at steps 2, 4, 6 and 8")


step  before_reset  spike  after_reset
   1         0.600      0        0.600
   2         1.080      1        0.000
   3         0.600      0        0.600
   4         1.080      1        0.000
   5         0.600      0        0.600
   6         1.080      1        0.000
   7         0.600      0        0.600
   8         1.080      1        0.000
TOTAL_SPIKES=4

PASS: spikes occur at steps 2, 4, 6 and 8


**Explain:** after the first step, the state is 0.6. After the second, it is `0.8 × 0.6 + 0.6 = 1.08`, so the neuron spikes and resets. The table shows state both before and after reset.

**Your experiment:** predict what happens if the threshold rises to 1.5. Run the next cell. Change only the threshold, leaving retention at 0.8.


In [4]:
higher_threshold = run_c("neuron", 1.5, 0.8)
if higher_threshold is not None:
    assert "TOTAL_SPIKES=2" in higher_threshold, "FAIL: expected two spikes"
    print("PASS: a higher threshold gives two spikes in this example")


step  before_reset  spike  after_reset
   1         0.600      0        0.600
   2         1.080      0        1.080
   3         1.464      0        1.464
   4         1.771      1        0.000
   5         0.600      0        0.600
   6         1.080      0        1.080
   7         1.464      0        1.464
   8         1.771      1        0.000
TOTAL_SPIKES=2

PASS: a higher threshold gives two spikes in this example


**Record:** baseline spikes = ___; higher-threshold spikes = ___; explain why in one sentence.

**Takeaway:** a spike is a discrete output event; internal state can accumulate between spikes.

## 2. What does the leak do?
**Problem:** recent evidence should matter more than old evidence.

**Predict:** keeping threshold at 1.0, what happens if retention is zero? Remember: retention is the fraction **kept**, so a smaller retention means more leakage.

Use the **same C program**, changing only retention. No new model code is needed.


In [5]:
no_memory = run_c("neuron", 1.0, 0.0)
if no_memory is not None:
    assert "TOTAL_SPIKES=0" in no_memory, "FAIL: expected no spikes"
    print("PASS: without retained state, each 0.6 input stays below threshold")


step  before_reset  spike  after_reset
   1         0.600      0        0.600
   2         0.600      0        0.600
   3         0.600      0        0.600
   4         0.600      0        0.600
   5         0.600      0        0.600
   6         0.600      0        0.600
   7         0.600      0        0.600
   8         0.600      0        0.600
TOTAL_SPIKES=0

PASS: without retained state, each 0.6 input stays below threshold


**Result and explanation:** with zero retention, every step begins without previous evidence. The state is always 0.6 before reset testing, so no spike occurs.

**Your experiment:** in the next cell, change retention from 0.5 to 0.9. Keep threshold at 1.0. Write the spike count for each run. These are observations to collect, not prefilled measurements.


In [6]:
retention = 0.5  # rerun with 0.9; leave threshold unchanged
student_result = run_c("neuron", 1.0, retention)


step  before_reset  spike  after_reset
   1         0.600      0        0.600
   2         0.900      0        0.900
   3         1.050      1        0.000
   4         0.600      0        0.600
   5         0.900      0        0.900
   6         1.050      1        0.000
   7         0.600      0        0.600
   8         0.900      0        0.900
TOTAL_SPIKES=2



**Takeaway:** the leak controls how strongly previous inputs influence the next output.

## 3. Send changes rather than repeated readings
**Problem:** a temperature sensor reports the same value repeatedly. A receiver needs to know its current value.

**Predict:** for `20,20,20,21,21,21,19,19`, how many messages are needed if we send the initial value, then only changes?

Read the C loop: a repeated sample keeps the last receiver value; a change updates it. The check ensures the receiver can reconstruct **every** sample in this noiseless example.


In [7]:
events_c = r"""#include <stdio.h>
int main(void) {
    const int samples[] = {20,20,20,21,21,21,19,19};
    const int n = sizeof(samples) / sizeof(samples[0]);
    int last = samples[0];
    int messages = 1;                 // send initial value
    int correct = 1;
    printf("Initial value: %d\n", last);
    for (int t = 1; t < n; t++) {
        if (samples[t] != last) {     // send only a change
            last = samples[t];
            messages++;
            printf("Change at t=%d: %d\n", t, last);
        }
        if (last != samples[t]) correct = 0;
    }
    printf("SAMPLES=%d MESSAGES=%d\n", n, messages);
    printf("%s: reconstructed readings\n", correct ? "PASS" : "FAIL");
    return correct ? 0 : 1;
}
"""
(work / "events.c").write_text(events_c)
print(events_c)


#include <stdio.h>
int main(void) {
    const int samples[] = {20,20,20,21,21,21,19,19};
    const int n = sizeof(samples) / sizeof(samples[0]);
    int last = samples[0];
    int messages = 1;                 // send initial value
    int correct = 1;
    printf("Initial value: %d\n", last);
    for (int t = 1; t < n; t++) {
        if (samples[t] != last) {     // send only a change
            last = samples[t];
            messages++;
            printf("Change at t=%d: %d\n", t, last);
        }
        if (last != samples[t]) correct = 0;
    }
    printf("SAMPLES=%d MESSAGES=%d\n", n, messages);
    printf("%s: reconstructed readings\n", correct ? "PASS" : "FAIL");
    return correct ? 0 : 1;
}



In [8]:
events = run_c("events")
if events is not None:
    assert "SAMPLES=8 MESSAGES=3" in events, "FAIL: unexpected message count"
    assert "PASS: reconstructed readings" in events
    print("PASS: three messages preserve eight readings")


Initial value: 20
Change at t=3: 21
Change at t=6: 19
SAMPLES=8 MESSAGES=3
PASS: reconstructed readings

PASS: three messages preserve eight readings


**Explain:** three messages carry the initial value and two changes. This illustrates sparse event communication. It is not a complete spiking network or an event-camera simulator. The CPU still examines all eight input samples.

**Your experiment:** change the C array to `{20,21,20,21,20,21,20,21}`. Rerun the source-writing cell and its run cell. Predict eight messages. The original three-message assertion will then fail intentionally: update its expected count to eight **only after explaining the changed input**.

**Takeaway:** event communication helps when inputs are sparse; frequent changes can remove the message-count advantage. Fewer messages alone do not prove lower runtime or energy. Hardware, routing and idle power also matter.

## 4. Explain what you learned
Before opening the answers, write a one-sentence response to each question.
1. Why did increasing the threshold reduce the spike count in our run?
2. What is the difference between state and a spike?
3. Why did zero retention prevent spikes here?
4. Would you choose event communication for an almost-constant sensor or for a sensor changing every sample? What else would you measure?

<details><summary>Expected reasoning — open after answering</summary>

1. More accumulated evidence was required before reset: four spikes became two.
2. State is a stored value; a spike is an output event when the threshold is reached.
3. Each 0.6 input was below threshold and no earlier state was retained.
4. The almost-constant stream has fewer messages. Measure complete-system latency, energy and correctness before claiming a hardware advantage.
</details>

## 5. Optional external practice — choose one
The core notebook above is complete without external software.

**A. Guided visual practice (10–15 minutes):** open [snnTorch Tutorial 2 — The Leaky Integrate-and-Fire Neuron](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_2.html). Use its own **Google Colab** link for an editable Python notebook and follow its setup there, not on SciTech. Read the neuron explanation, run a supplied example and change one input-strength setting. Save one before/after plot and explain the spike change. The tutorial uses a different parameterisation/reset convention: do not expect identical counts to our C model. A Google account/runtime may be required; if unavailable, read the figures and annotate them instead.

**B. Extra reading and exercises:** use [EPFL Neuronal Dynamics exercises](https://neuronaldynamics.epfl.ch/exercises.html), follow the exercise-site link and choose the leaky integrate-and-fire section. Predict how an input-current change affects firing, then compare with its example. Its Python dependencies are optional and unnecessary for this lesson; use an existing personal environment if you choose to execute it.

**Submit:** the parameter changed, your prediction, one observed output/annotated figure, and two sentences explaining it. Label external results separately from the C simulations.

## Validation and limits
The saved outputs are from **LOCAL_CPU** execution. The C compilation and numerical/output checks passed locally. Real SciTech execution and JupyterLab URL import remain **NOT RUN**. Neuromorphic hardware execution is **SKIPPED** because this notebook uses CPU simulations only. No GPU, MPI, training benchmark or energy measurement is included.
